In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

from load_data import get_spark, load_ratings

spark = get_spark()
spark.sparkContext.setLogLevel("WARN")

ratings_sdf = load_ratings(spark)
ratings = ratings_sdf.toPandas()

print(ratings.shape)
ratings.head()

C:\Users\Iqra\OneDrive\Desktop\Movie_Recomm_Engine\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\Iqra\OneDrive\Desktop\Movie_Recomm_Engine\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


(1000209, 4)


,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [2]:
ratings_sorted = ratings.sort_values(["user_id", "timestamp"])

# For each user, hold out their last 20% of ratings (by time) as test set
def split_user(group):
    n = len(group)
    n_test = max(1, int(n * 0.2))
    return group.iloc[:-n_test], group.iloc[-n_test:]

train_parts = []
test_parts = []

for _, group in ratings_sorted.groupby("user_id"):
    tr, te = split_user(group)
    train_parts.append(tr)
    test_parts.append(te)

train_df = pd.concat(train_parts).reset_index(drop=True)
test_df = pd.concat(test_parts).reset_index(drop=True)

print(f"Train: {len(train_df):,} rows ({len(train_df)/len(ratings)*100:.1f}%)")
print(f"Test:  {len(test_df):,} rows ({len(test_df)/len(ratings)*100:.1f}%)")

Train: 802,553 rows (80.2%)
Test:  197,656 rows (19.8%)


In [3]:
# Build encoders from the FULL dataset so train/test share the same ID space
unique_users = np.sort(ratings["user_id"].unique())
unique_movies = np.sort(ratings["movie_id"].unique())

user_to_idx = {u: i for i, u in enumerate(unique_users)}
movie_to_idx = {m: i for i, m in enumerate(unique_movies)}

n_users = len(unique_users)
n_movies = len(unique_movies)
print(f"n_users: {n_users}, n_movies: {n_movies}")

for df in (train_df, test_df):
    df["user_idx"] = df["user_id"].map(user_to_idx)
    df["movie_idx"] = df["movie_id"].map(movie_to_idx)

train_df.head()

n_users: 6040, n_movies: 3706


,user_id,movie_id,rating,timestamp,user_idx,movie_idx
0,1,3186,4,978300019,0,2969
1,1,1270,5,978300055,0,1178
2,1,1721,4,978300055,0,1574
3,1,1022,5,978300055,0,957
4,1,2340,3,978300103,0,2147


In [4]:
train_matrix = csr_matrix(
    (train_df["rating"], (train_df["user_idx"], train_df["movie_idx"])),
    shape=(n_users, n_movies)
)

print(f"Matrix shape: {train_matrix.shape}")
print(f"Non-zero entries: {train_matrix.nnz:,}")
print(f"Density: {train_matrix.nnz / (n_users * n_movies) * 100:.3f}%")
print(f"Memory (sparse): {train_matrix.data.nbytes / 1024:.1f} KB")
print(f"Memory (if dense): {n_users * n_movies * 8 / 1024 / 1024:.1f} MB")

Matrix shape: (6040, 3706)
Non-zero entries: 802,553
Density: 3.585%
Memory (sparse): 3135.0 KB
Memory (if dense): 170.8 MB


In [5]:
import pickle

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train_df.to_parquet(PROCESSED_DIR / "train.parquet", index=False)
test_df.to_parquet(PROCESSED_DIR / "test.parquet", index=False)

with open(PROCESSED_DIR / "encoders.pkl", "wb") as f:
    pickle.dump({
        "user_to_idx": user_to_idx,
        "movie_to_idx": movie_to_idx,
        "n_users": n_users,
        "n_movies": n_movies,
    }, f)

print("Saved train.parquet, test.parquet, encoders.pkl to data/processed/")

Saved train.parquet, test.parquet, encoders.pkl to data/processed/
